# Train

> Some Specific Trainers

In [ ]:
#| default_exp train

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import torch, torch.nn as nn, torch.nn.functional as F, lightning.pytorch as pl, warnings, math

from torch.optim.lr_scheduler import OneCycleLR, CosineAnnealingWarmRestarts

from sleepjepa.loss import FocalLoss, KLDivLoss, CrossEntropyLoss

In [ ]:
#| export
class PatchTFTSleepStage(pl.LightningModule):
    def __init__(self, 
                 learning_rate,
                 train_size,
                 batch_size,
                 n_gpus,
                 linear_probing_head,
                 preloaded_model,
                 metrics={},
                 fine_tune=False,
                 loss_fxn='CrossEntropy',
                 class_weights=None,
                 gamma=2.,
                 label_smoothing=0,
                 y_padding_mask=-100,
                 epochs=100,
                 weight_decay=0.,
                 use_weight_decay_scheduler=False,
                 final_weight_decay=0.01,
                 optimizer_type='Adam',
                 scheduler_type='OneCycle',
                 scheduler_kwargs={},
                 ):
        super().__init__()
        self.encoder = preloaded_model
        assert loss_fxn.lower() in ['crossentropy', 'focalloss', 'kldivloss'], "loss_fxn must be either CrossEntropy or FocalLoss or KLDivLoss"
        
        self.scheduler_type = scheduler_type
        if self.scheduler_type is not None:
            assert self.scheduler_type.lower() in ['onecycle', 'cosineannealingwarmrestarts'], "scheduler must be either OneCycle, CosineAnnealingWarmRestarts, or None"
        self.weight_decay = weight_decay
        self.label_smoothing = label_smoothing
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.train_size = train_size
        self.batch_size = batch_size * n_gpus
        self.ipe = self.train_size//self.batch_size
        self.metrics = nn.ModuleDict(metrics)
        self.y_padding_mask = y_padding_mask
        self.ipe = self.train_size//self.batch_size
        self.class_weights = class_weights
        self.loss_fxn = loss_fxn
        self.gamma = gamma
        self.fine_tune = fine_tune
        self.use_weight_decay_scheduler = use_weight_decay_scheduler
        self.final_weight_decay = final_weight_decay
        self.scheduler_kwargs = scheduler_kwargs

        if not self.fine_tune:
            self.encoder.freeze()
            if hasattr(self.encoder, 'pretrain'):
                setattr(self.encoder, 'pretrain', False)
                setattr(self.encoder.model, 'pretrain', False)
        self.feedforward = linear_probing_head 
        self.optimizer_type = optimizer_type
        
        self.save_hyperparameters(ignore=['linear_probing_head', 'preloaded_model'])

    def forward(self, x):
        """
        In: [bs, n_channels, seq_len]
        """
        x = self.encoder(x)
        if isinstance(x, tuple):
            x = x[0]
        x = self.feedforward(x)
        return x
    
    def on_train_batch_start(self, batch, batch_idx):
        if self.use_weight_decay_scheduler:
            step = self.global_step
            T_max = int(self.ipe * self.epochs)
            progress = step / T_max
            new_wd = self.final_weight_decay + (self.weight_decay - self.final_weight_decay) * 0.5 * (1. + math.cos(math.pi * progress))

            if self.final_weight_decay <= self.weight_decay:
                new_wd = max(self.final_weight_decay, new_wd)
            else:
                new_wd = min(self.final_weight_decay, new_wd)

            for group in self.optimizer.param_groups:
                if ('WD_exclude' not in group) or not group['WD_exclude']:
                    group['weight_decay'] = new_wd

    def predict_step(self, batch, batch_idx, dataloader_idx=0):
        x,y, *idx = batch
        x = self.encoder(x)
        if isinstance(x, tuple):
            x = x[0]
        preds = self.feedforward(x)
        return preds, y
    
    def training_step(self, batch, batch_idx):
        x, y, *idx = batch
        x = self.encoder(x)
        if isinstance(x, tuple):
            x = x[0]
        x = self.feedforward(x) 
        if x.is_nested:
            x = x.to_padded_tensor(padding=0)
        if y.is_nested:
            y = y.to_padded_tensor(padding=self.y_padding_mask, output_size=(x.shape[0], x.shape[2]))
        if self.loss_fxn.lower() == 'kldivloss':
            loss = KLDivLoss(weight=self.class_weights.to(x.device) if self.class_weights is not None else None, ignore_index=self.y_padding_mask)(x,y)
        else:
            ce_loss = CrossEntropyLoss(weight=self.class_weights.to(x.device) if self.class_weights is not None else None, label_smoothing=self.label_smoothing, ignore_index=self.y_padding_mask)
            focal_loss = FocalLoss(weight=self.class_weights.to(x.device) if self.class_weights is not None else None, gamma=self.gamma, ignore_index=self.y_padding_mask)
            loss = ce_loss(x,y) if self.loss_fxn == 'CrossEntropy' else focal_loss(x,y)
            self.log('train_ce_loss' if self.loss_fxn != 'CrossEntropy' else 'train_focal_loss', ce_loss(x,y) if self.loss_fxn != 'CrossEntropy' else focal_loss(x,y), prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        loss = loss.to(x.device)
        self.log('train_loss', loss, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        
        if batch_idx == 0:
            print("Head parameters and requires_grad:")
            for name, param in self.feedforward.named_parameters():
                print(name, param.requires_grad, param.grad.abs().mean() if param.grad is not None else None)
            print("Head output shape:", x.shape)
            print("Unique targets:", torch.unique(y, return_counts=True))
            print("Unique predicted classes:", torch.unique(torch.argmax(x, dim=1), return_counts=True))
            print("Predicted classes (first sample):", torch.argmax(x[0], dim=0)[:20])
            print("Target classes (first sample):", y[0][:20])
        return loss
    
    def validation_step(self, batch, batch_idx):
        x, y, *idx = batch
        x = self.encoder(x)
        if isinstance(x, tuple):
            x = x[0]
        x = self.feedforward(x)
        if x.is_nested:
            x = x.to_padded_tensor(padding=0)
        if y.is_nested:
            y = y.to_padded_tensor(padding=self.y_padding_mask, output_size=(x.shape[0], x.shape[2]))
        if self.loss_fxn.lower() == 'kldivloss':
            loss = KLDivLoss(weight=self.class_weights.to(x.device) if self.class_weights is not None else None, ignore_index=self.y_padding_mask)(x,y)
        else:
            ce_loss = CrossEntropyLoss(weight=self.class_weights.to(x.device) if self.class_weights is not None else None, label_smoothing=self.label_smoothing, ignore_index=self.y_padding_mask)
            focal_loss = FocalLoss(weight=self.class_weights.to(x.device) if self.class_weights is not None else None, gamma=self.gamma, ignore_index=self.y_padding_mask)
            loss = ce_loss(x,y) if self.loss_fxn == 'CrossEntropy' else focal_loss(x,y)
            self.log('val_ce_loss' if self.loss_fxn != 'CrossEntropy' else 'val_focal_loss', ce_loss(x,y) if self.loss_fxn != 'CrossEntropy' else focal_loss(x,y), prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        self.log('val_loss', loss, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        x_probs = torch.softmax(x, dim=1)
        for metric in self.metrics:
            self.metrics[metric].update(x_probs, y)

    def on_validation_epoch_start(self):
        torch.cuda.empty_cache()
    
    def on_validation_epoch_end(self):
        for name, metric in self.metrics.items():
            self.log(f'val_{name}', metric.compute(), prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
            metric.reset()
        
    def configure_optimizers(self):
        param_groups = [
           {
                'params': (p for n, p in self.feedforward.named_parameters()
                        if ('bias' not in n) and (len(p.shape) != 1))
            }, {
                'params': (p for n, p in self.feedforward.named_parameters()
                        if ('bias' in n) or (len(p.shape) == 1)),
                'WD_exclude': True,
                'weight_decay': 0,
            },
        ]
        self.optimizer = torch.optim.AdamW(param_groups, lr=self.learning_rate, weight_decay=self.weight_decay if not self.use_weight_decay_scheduler else 0) if self.optimizer_type.lower() == 'adamw' else\
                     torch.optim.Adam(param_groups, lr=self.learning_rate, weight_decay=self.weight_decay if not self.use_weight_decay_scheduler else 0)
        if self.scheduler_type.lower() == 'onecycle':
            scheduler = OneCycleLR(self.optimizer, total_steps=self.trainer.estimated_stepping_batches, **self.scheduler_kwargs)
            lr_scheduler = {'scheduler': scheduler, 'interval': 'step'}
            return {'optimizer': self.optimizer, 'lr_scheduler': lr_scheduler}
        elif self.scheduler_type.lower() == 'cosineannealingwarmrestarts':
            scheduler = CosineAnnealingWarmRestarts(self.optimizer, **self.scheduler_kwargs)
            lr_scheduler = {'scheduler': scheduler, 'interval': 'epoch'}
            return {'optimizer': self.optimizer, 'lr_scheduler': lr_scheduler}
        else:
            return self.optimizer

In [ ]:
#| export
class PatchTFTSingleOutcomeLightning(pl.LightningModule): 
    def __init__(self, 
                linear_probing_head,
                learning_rate,
                train_size,
                batch_size,
                n_gpus,
                preloaded_model,
                metrics={},
                fine_tune=False,
                class_weights=None,
                epochs=100,
                scheduler_type='OneCycle',
                optimizer_type='AdamW',
                weight_decay=0.,
                use_weight_decay_scheduler=False,
                final_weight_decay=0.01,
                scheduler_kwargs={},
                transforms=None,
                mixup_callback=None,
                regression=False,
                loss_func=None,
                ):
        super().__init__()
        self.encoder = preloaded_model
        self.scheduler_type = scheduler_type
        if self.scheduler_type is not None:
            assert self.scheduler_type.lower() in ['onecycle', 'cosineannealingwarmrestarts'], "scheduler must be either OneCycle, CosineAnnealingWarmRestarts, or None"
        self.weight_decay = weight_decay
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.train_size = train_size
        self.batch_size = batch_size * n_gpus
        self.ipe = self.train_size//self.batch_size
        self.metrics = nn.ModuleDict(metrics)
        self.ipe = self.train_size//self.batch_size
        self.class_weights = class_weights
        self.fine_tune = fine_tune
        self.use_weight_decay_scheduler = use_weight_decay_scheduler
        self.scheduler_kwargs = scheduler_kwargs
        self.final_weight_decay = final_weight_decay
        self.optimizer_type = optimizer_type
        self.transforms = transforms
        self.mixup_callback = mixup_callback
        self.regression = regression
        self.loss_func = loss_func
        if not self.fine_tune:
            self.encoder.freeze()
            if hasattr(self.encoder, 'pretrain'):
                setattr(self.encoder, 'pretrain', False)
                setattr(self.encoder.model, 'pretrain', False)
        self.feedforward = linear_probing_head
        self.save_hyperparameters(ignore=['linear_probing_head', 'preloaded_model'])

    def forward(self, x):
        x = self.encoder(x)
        if isinstance(x, tuple):
            x = x[0]
        if torch.isnan(x).any():
            warnings.warn("NaN values in input to feedforward layer")
        x = self.feedforward(x)
        return x
    
    def on_train_batch_start(self, batch, batch_idx):
        if self.use_weight_decay_scheduler:
            step = self.global_step
            T_max = int(self.ipe * self.epochs)
            progress = step / T_max
            new_wd = self.final_weight_decay + (self.weight_decay - self.final_weight_decay) * 0.5 * (1. + math.cos(math.pi * progress))

            if self.final_weight_decay <= self.weight_decay:
                new_wd = max(self.final_weight_decay, new_wd)
            else:
                new_wd = min(self.final_weight_decay, new_wd)

            for group in self.optimizer.param_groups:
                if ('WD_exclude' not in group) or not group['WD_exclude']:
                    group['weight_decay'] = new_wd

    def predict_step(self, batch, batch_idx, dataloader_idx=0):
        x,y = batch
        preds = self(x)
        return preds, y

    def training_step(self, batch, batch_idx):
        if self.transforms is not None:
            batch = self.transforms(batch)
        if self.mixup_callback is not None:
            batch = self.mixup_callback(batch)
        x, y = batch
        x = self(x)
        if self.regression:
            loss = nn.MSELoss() if self.loss_func is None else self.loss_func
            loss_val = loss(x.squeeze(), y.squeeze().float())
        else:
            loss = nn.BCEWithLogitsLoss(pos_weight=self.class_weights.to(x.device) if self.class_weights is not None else None) if self.loss_func is None else self.loss_func
            loss_val = loss(x.squeeze(),y.squeeze().float())
        self.log('train_loss', loss_val, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        return loss_val

    def validation_step(self, batch, batch_idx):
        x, y = batch
        x = self(x)
        if self.regression:
            loss = nn.MSELoss() if self.loss_func is None else self.loss_func
            loss_val = loss(x.squeeze(), y.squeeze().float())
        else:
            loss = nn.BCEWithLogitsLoss(pos_weight=self.class_weights.to(x.device) if self.class_weights is not None else None) if self.loss_func is None else self.loss_func
            loss_val = loss(x.squeeze(),y.squeeze().float())
        self.log('val_loss', loss_val, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        
        x_probs = torch.sigmoid(x)
        for metric in self.metrics:
            self.metrics[metric].update(x_probs.squeeze(), y.squeeze().long())
    
    def on_validation_epoch_end(self):
        all_score = 0
        for name, metric in self.metrics.items():
            metric_val = metric.compute()
            all_score += metric_val
            self.log(f'val_{name}', metric_val, prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
            metric.reset()
        self.log('val_all_score', all_score, prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
    
    def on_validation_epoch_start(self):
        torch.cuda.empty_cache()

    def configure_optimizers(self):
        param_groups = [
           {
                'params': (p for n, p in self.feedforward.named_parameters()
                        if ('bias' not in n) and (len(p.shape) != 1))
            }, {
                'params': (p for n, p in self.feedforward.named_parameters()
                        if ('bias' in n) or (len(p.shape) == 1)),
                'WD_exclude': True,
                'weight_decay': 0,
            },
        ]
        self.optimizer = torch.optim.AdamW(param_groups, lr=self.learning_rate, weight_decay=self.weight_decay if not self.use_weight_decay_scheduler else 0) if self.optimizer_type.lower() == 'adamw' else\
                     torch.optim.Adam(param_groups, lr=self.learning_rate, weight_decay=self.weight_decay if not self.use_weight_decay_scheduler else 0)
        if self.scheduler_type.lower() == 'onecycle':
            scheduler = OneCycleLR(self.optimizer, total_steps=self.trainer.estimated_stepping_batches, **self.scheduler_kwargs)
            lr_scheduler = {'scheduler': scheduler, 'interval': 'step'}
            return {'optimizer': self.optimizer, 'lr_scheduler': lr_scheduler}
        elif self.scheduler_type.lower() == 'cosineannealingwarmrestarts':
            scheduler = CosineAnnealingWarmRestarts(self.optimizer, **self.scheduler_kwargs)
            lr_scheduler = {'scheduler': scheduler, 'interval': 'epoch'}
            return {'optimizer': self.optimizer, 'lr_scheduler': lr_scheduler}
        else:
            return self.optimizer

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()